In [15]:
# ============================================
# NETWORK INTRUSION DETECTION SYSTEM
# ============================================

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings

# Suppress warnings for clean output
warnings.filterwarnings("ignore")

# ============================================
# 1. LOAD DATASET
# ============================================

train = pd.read_csv(
    r"C:\Users\ASUS\Documents\cyber_security\Network_Intrusion_Detection_System\train_data.csv",
    header=None,
    sep=r'\s+'  # whitespace separator
)

test = pd.read_csv(
    r"C:\Users\ASUS\Documents\cyber_security\Network_Intrusion_Detection_System\test_data.csv",
    header=None,
    sep=r'\s+'
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# ============================================
# 2. DEFINE COLUMN NAMES
# ============================================

train_columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
'num_compromised','root_shell','su_attempted','num_root',
'num_file_creations','num_shells','num_access_files',
'num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate',
'diff_srv_rate','srv_diff_host_rate','dst_host_count',
'dst_host_srv_count','dst_host_same_srv_rate',
'dst_host_diff_srv_rate','dst_host_same_src_port_rate',
'dst_host_srv_diff_host_rate','dst_host_serror_rate',
'dst_host_srv_serror_rate','dst_host_rerror_rate',
'dst_host_srv_rerror_rate','label','difficulty'
]

test_columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
'num_compromised','root_shell','su_attempted','num_root',
'num_file_creations','num_shells','num_access_files',
'num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate',
'diff_srv_rate','srv_diff_host_rate','dst_host_count',
'dst_host_srv_count','dst_host_same_srv_rate',
'dst_host_diff_srv_rate','dst_host_same_src_port_rate',
'dst_host_srv_diff_host_rate','dst_host_serror_rate',
'dst_host_srv_serror_rate','dst_host_rerror_rate',
'dst_host_srv_rerror_rate','label'
]

# ============================================
# 3. ASSIGN COLUMN NAMES
# ============================================

train.columns = train_columns
test.columns = test_columns

# ============================================
# 4. REMOVE DIFFICULTY COLUMN FROM TRAIN
# ============================================

train.drop('difficulty', axis=1, inplace=True)

# ============================================
# 5. CLEAN LABELS AND CONVERT TO BINARY
# ============================================

train['label'] = train['label'].str.strip().apply(lambda x: 0 if x=='normal' else 1)
test['label']  = test['label'].str.strip().apply(lambda x: 0 if x=='normal' else 1)

print("Train label distribution:\n", train['label'].value_counts())
print("Test label distribution:\n", test['label'].value_counts())

# ============================================
# 6. ENCODE CATEGORICAL FEATURES
# ============================================

categorical_cols = ['protocol_type', 'service', 'flag']

for col in categorical_cols:
    encoder = LabelEncoder()
    combined = pd.concat([train[col], test[col]])
    encoder.fit(combined)
    train[col] = encoder.transform(train[col])
    test[col] = encoder.transform(test[col])

# ============================================
# 7. SPLIT FEATURES AND LABELS
# ============================================

X_train = train.drop('label', axis=1)
y_train = train['label']

X_test = test.drop('label', axis=1)
y_test = test['label']

# ============================================
# 8. TRAIN RANDOM FOREST MODEL
# ============================================

model = RandomForestClassifier(n_estimators=100, random_state=42)
print("\nTraining Model...")
model.fit(X_train, y_train)

# ============================================
# 9. PREDICT ATTACKS
# ============================================

print("\nDetecting Intrusions...")
y_pred = model.predict(X_test)

# ============================================
# 10. MODEL EVALUATION
# ============================================

print("\nModel Performance")
print("----------------------")
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================
# 11. SAMPLE PREDICTION
# ============================================

sample = X_test.iloc[[0]]  # keep as DataFrame to avoid warnings
prediction = model.predict(sample)

if prediction[0] == 1:
    print("\nPrediction: ATTACK DETECTED")
else:
    print("\nPrediction: NORMAL TRAFFIC")

Train shape: (125973, 43)
Test shape: (22544, 42)
Train label distribution:
 label
0    67343
1    58630
Name: count, dtype: int64
Test label distribution:
 label
1    22544
Name: count, dtype: int64

Training Model...

Detecting Intrusions...

Model Performance
----------------------
Accuracy: 0.36142654364797727

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.36      0.53     22544

    accuracy                           0.36     22544
   macro avg       0.50      0.18      0.27     22544
weighted avg       1.00      0.36      0.53     22544


Prediction: ATTACK DETECTED
